# 프로젝트 1. 금융·신용 데이터 분석

- 작성: 양재일 (서울사이버대학교 인공지능학과)
- 목적: 공개 신용 데이터로 연체·부실과 관련된 특성을 살펴보고, 여신 실무 관점에서 결과를 해석한다.
- 사용 도구: Python, pandas, seaborn, scikit-learn
- 데이터: [OpenML German Credit (`credit-g`)](https://www.openml.org/d/31) — 개인·고객 식별 정보 없음, 공개 데이터만 사용

> 이 노트북은 학습용입니다. 실제 여신 심사 모형이나 개인 신용평가에 사용할 수 없습니다.

## 1. 문제 정의

은행 지점과 기업여신 현장에서 반복해서 확인하던 질문은 단순합니다.

> **이 차주가 원리금을 약정대로 갚을 가능성이 얼마나 되는가?**

실무에서는 재무제표·담보·보증·거래실적과 함께, 아래와 같은 정성·정량 항목을 같이 봅니다.

| 실무에서 보던 항목 | 이 데이터에서 대응하는 변수 |
| --- | --- |
| 예금·결제성 잔액 | 당좌예금 상태(`checking_status`) |
| 과거 상환 이력 | 신용 이력(`credit_history`) |
| 자금 용도 | 대출 목적(`purpose`) |
| 대출 규모·기간 | 금액(`credit_amount`), 기간(`duration`) |
| 저축·재산 | 저축(`savings_status`), 재산(`property_magnitude`) |
| 소득의 안정성 | 근속기간(`employment`), 직업(`job`) |

이 프로젝트에서는 공개된 1,000건의 독일 신용 표본으로

1. 부실(bad) 비율이 높은 집단을 찾고
2. 간단한 분류 모델로 변수 중요도를 확인한 뒤
3. 결과를 여신 실무 언어로 해석합니다.

## 2. 데이터

- 출처: Hofmann, H. (1994). *Statlog (German Credit Data)*. UCI / OpenML `credit-g`
- 표본: 1,000건, 특성 20개, 목표 변수 `class` (`good` / `bad`)
- 라이선스: 공개 연구·교육 목적
- 주의: 1990년대 독일 소매 신용 자료이므로 한국 현재 여신 환경과 수치를 직접 비교하지 않습니다. **해석의 틀**(어떤 정보가 위험을 설명하는가)을 배우는 데 사용합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay,
)

import os, sys, subprocess
from pathlib import Path
from matplotlib import font_manager as fm

def setup_korean_font():
    font_path = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    if sys.platform.startswith("linux") and not font_path.exists():
        subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False, capture_output=True)
    if font_path.exists():
        fm.fontManager.addfont(str(font_path))
        plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False

setup_korean_font()
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)
RANDOM_STATE = 42
print("라이브러리 준비 완료")

In [ ]:
dataset = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
df = dataset.frame.copy()

COLUMN_KO = {
    "checking_status": "당좌예금상태",
    "duration": "대출기간_월",
    "credit_history": "신용이력",
    "purpose": "대출목적",
    "credit_amount": "대출금액",
    "savings_status": "저축상태",
    "employment": "근속기간",
    "installment_commitment": "할부부담률",
    "personal_status": "개인상태",
    "other_parties": "보증인",
    "residence_since": "거주기간",
    "property_magnitude": "재산",
    "age": "나이",
    "other_payment_plans": "기타할부",
    "housing": "주거형태",
    "existing_credits": "기존대출건수",
    "job": "직업",
    "num_dependents": "부양가족",
    "own_telephone": "전화보유",
    "foreign_worker": "외국인근로자",
    "class": "신용결과",
}
df = df.rename(columns=COLUMN_KO)
df["부실여부"] = (df["신용결과"] == "bad").astype(int)

print("행, 열:", df.shape)
print(df["신용결과"].value_counts())
print("부실 비율: {:.1f}%".format(df["부실여부"].mean() * 100))
df.head()

## 3. 방법

1. **탐색적 분석(EDA)**: 금액·기간·나이 분포와, 범주별 부실 비율을 비교한다.
2. **교차 확인**: 당좌예금 상태 × 신용 이력처럼 실무에서 같이 보던 조합을 표로 본다.
3. **분류 모형**: 로지스틱 회귀로 부실 여부를 예측하고, 계수 부호로 위험·완화 요인을 읽는다.
4. **평가**: 정확도만 보지 않고 혼동행렬·AUC를 확인한다. 여신에서는 **부실을 놓치는 것(미탐)**이 더 큰 손실이기 때문이다.

In [ ]:
print(df[["대출기간_월", "대출금액", "나이", "할부부담률"]].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["대출금액", "대출기간_월", "나이"]):
    sns.boxplot(data=df, x="신용결과", y=col, ax=ax, palette={"good": "#4c9a8a", "bad": "#c45c4c"})
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
def bad_rate_table(column):
    table = (
        df.groupby(column)["부실여부"]
        .agg(건수="count", 부실건수="sum", 부실비율="mean")
        .sort_values("부실비율", ascending=False)
    )
    table["부실비율"] = (table["부실비율"] * 100).round(1)
    return table

print("=== 당좌예금상태별 부실 비율 ===")
display(bad_rate_table("당좌예금상태"))
print("=== 신용이력별 부실 비율 ===")
display(bad_rate_table("신용이력"))
print("=== 저축상태별 부실 비율 ===")
display(bad_rate_table("저축상태"))

In [ ]:
key_cols = ["당좌예금상태", "신용이력", "저축상태", "근속기간", "재산", "대출목적"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), key_cols):
    rates = df.groupby(col)["부실여부"].mean().sort_values(ascending=False) * 100
    rates.plot(kind="bar", ax=ax, color="#bd5d38")
    ax.set_ylabel("부실 비율(%)")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=70)
plt.tight_layout()
plt.show()

In [ ]:
cross = pd.crosstab(df["당좌예금상태"], df["신용이력"], values=df["부실여부"], aggfunc="mean") * 100
plt.figure(figsize=(9, 4.5))
sns.heatmap(cross.round(1), annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("당좌예금상태 × 신용이력 부실 비율(%)")
plt.show()

## 4. 분류 모형 (로지스틱 회귀)

트리 모형보다 계수의 부호를 읽기 쉬운 로지스틱 회귀를 사용합니다.  
범주형은 원-핫 인코딩, 숫자는 표준화합니다. 학습·평가는 7:3으로 나눕니다.

In [ ]:
y = df["부실여부"]
X = df.drop(columns=["신용결과", "부실여부"])

cat_cols = X.select_dtypes(include=["category", "object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["category", "object"]).columns.tolist()

preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

model = Pipeline(
    [
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("AUC: {:.3f}".format(roc_auc_score(y_test, y_prob)))
print(classification_report(y_test, y_pred, target_names=["정상(good)", "부실(bad)"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=["예측 정상", "예측 부실"],
    yticklabels=["실제 정상", "실제 부실"],
)
axes[0].set_title("혼동행렬")
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[1])
axes[1].set_title("ROC 곡선")
plt.tight_layout()
plt.show()

In [ ]:
ohe = model.named_steps["prep"].named_transformers_["cat"]
feat_names = np.concatenate([num_cols, ohe.get_feature_names_out(cat_cols)])
coef = model.named_steps["clf"].coef_[0]
imp = pd.Series(coef, index=feat_names).sort_values()

top = pd.concat([imp.head(8), imp.tail(8)])
colors = ["#4c9a8a" if v < 0 else "#c45c4c" for v in top.values]
plt.figure(figsize=(8, 6))
top.plot(kind="barh", color=colors)
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("로지스틱 회귀 계수 (양수=부실 확률 증가)")
plt.tight_layout()
plt.show()

print("부실 확률을 높이는 요인 (상위 5)")
print(imp.sort_values(ascending=False).head(5).round(3))
print("\n부실 확률을 낮추는 요인 (상위 5)")
print(imp.sort_values().head(5).round(3))

## 5. 결과 해석 (여신 실무 관점)

숫자는 실행 결과에 따라 조금 달라질 수 있습니다. 방향은 대체로 아래와 같습니다.

1. **결제성 예금이 없거나 마이너스인 집단**의 부실 비율이 높다.  
   실무에서 급여이체·평균잔액을 보던 이유와 같다. 현금흐름이 안 보이면 상환 여력을 확인하기 어렵다.
2. **신용 이력이 불량하거나 기존 연체가 있는 경우**가 가장 강한 위험 신호다.  
   과거 상환 태도는 미래 상환의 가장 직접적인 근거였다.
3. **대출 기간이 길고 금액이 클수록** 부실 가능성이 커지는 경향이 있다.  
   장기·고액 여신은 금리·분할 상환 설계와 담보 보완이 필요했던 이유와 맞는다.
4. **저축·재산이 있는 차주**는 상대적으로 부실이 낮다.  
   담보·예적금 질권과 같은 ‘완충 자산’의 역할이다.
5. 모형 AUC는 보통 0.7 전후다. 실무 심사자를 대체할 수준이 아니라,  
   **어떤 항목을 먼저 볼지 우선순위를 정하는 보조 도구**로 쓰는 것이 맞다.

혼동행렬에서 실제 부실을 정상으로 분류한 칸은, 현장에서 말하면 **회수 불능 여신을 실행한 경우**에 가깝다. 학습 모형이라도 이 칸을 줄이는 쪽이 더 중요하다.

## 6. 배운 점과 한계

**배운 점**
- 여신 심사의 정성 항목을 데이터 변수로 바꾸면, 집단별 부실 비율을 숫자로 비교할 수 있다.
- pandas 집계와 로지스틱 회귀 계수만으로도 ‘위험 요인 / 완화 요인’을 분리해 설명할 수 있다.
- 분류 문제는 정확도 한 줄이 아니라 미탐(부실을 정상으로 보는 오류)을 같이 봐야 한다.

**한계**
- 표본이 오래되고 국가·상품이 다르다. 한국 소상공인·기업여신에 바로 적용할 수 없다.
- 담보 가치, 보증인 자력, 업황, 대표자 신용은 이 데이터에 거의 없다.
- 원-핫 인코딩 계수는 범주가 많을수록 해석이 분산된다. 다음 단계에서는 범주를 실무 기준으로 묶을 수 있다.

**다음 개선**
- 범주를 ‘양호 / 보통 / 미흡’처럼 심사 등급으로 재코딩한다.
- 랜덤 포레스트와 성능·해석을 비교한다.
- 프로젝트 2(자영업자 재무비율)와 연결해, 개인 신용 + 사업 재무를 같이 보는 구조를 만든다.